In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [ ]:
# Tratamiento de Datos
import pandas as pd
import numpy as np
from IPython.display import display

# Visualizaciones
import matplotlib.pyplot as plt
import seaborn as sns

# Para que se muestren todas las columnas al inspeccionar los DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)

In [ ]:
from framework import sp_carga_exploracion as sc
from framework import sp_limpieza_transformacion as sl
from framework import sp_eda as se
from framework import sp_modelado as sm
from framework import sp_clustering as cl

In [ ]:
df = sc.leer_csv('../data/processed/02_datos_limpios.csv')
df_e = df.copy()

1️⃣ Las categóricas necesitan codificar_categoricas() antes de estandarizar_clustering()

estandarizar_clustering() solo sabe trabajar con números — si le pasas nivel_dificultad tal cual (texto), fallaría o la ignoraría. Así que reutilizamos el paso que ya conoces de sp_modelado.py antes de entrar en sp_clustering.py:

In [ ]:
df_cluster = sm.codificar_categoricas(df_e, ['nivel_dificultad', 'horario_estudio_preferido', 'estilo_aprendizaje'])
df_cluster = df_cluster.drop(columns=['tiene_tutor'])   # nos quedamos con tiene_tutor_ml, como en modelado

2️⃣ Aviso rápido sobre incluir aprobado + nota_final juntas

Como ya sabes, son la misma información dos veces (una se deriva de la otra). Para clustering no es "leakage" en el sentido estricto (no hay nada que predecir), pero sí puede hacer que el algoritmo agrupe en parte "por lo mismo dos veces" — dándole más peso implícito al rendimiento académico que al resto de variables. Como es para aprender, lo dejamos así y lo comentamos cuando veamos los resultados — es un buen ejercicio ver el efecto real, en vez de solo que te lo cuente.

Paso 1: preparación (estandarizar_clustering())

In [ ]:
X_cluster, scaler_cluster = cl.estandarizar_clustering(df_cluster)
X_cluster.shape

Aquí no hace falta train_test() — a diferencia de regresión/clasificación, clustering no predice sobre datos nuevos en el mismo sentido, así que se usa el dataset completo (aunque a veces sí se reserva una muestra para validar clusters en producción, no es lo habitual en un ejercicio de curso).